In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import pickle
from google.colab import drive

drive.mount('/content/drive')

with open("/content/drive/MyDrive/SmartRail/time_split_data.pkl", "rb") as f:
    data = pickle.load(f)

X_train_time = data["X_train_time"]
y_train_time = data["y_train_time"]
X_test_time = data["X_test_time"]
y_test_time = data["y_test_time"]

log_reg = joblib.load("/content/drive/MyDrive/SmartRail/log_reg.pkl")
rf = joblib.load("/content/drive/MyDrive/SmartRail/random_forest.pkl")
xgb_tuned = joblib.load("/content/drive/MyDrive/SmartRail/xgboost_tuned_final.pkl")
scaler2 = joblib.load("/content/drive/MyDrive/SmartRail/scaler2.pkl")

print("All loaded — ready for stacking")

Mounted at /content/drive
All loaded — ready for stacking


In [2]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report

estimators = [
    ('lr', log_reg),
    ('rf', rf),
    ('xgb', xgb_tuned)
]

stack_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=3,
    n_jobs=-1
)

stack_model.fit(X_train_time, y_train_time)

y_prob_stack = stack_model.predict_proba(X_test_time)[:, 1]
auc_stack = roc_auc_score(y_test_time, y_prob_stack)
print(f"Stacking AUC: {auc_stack:.4f}")

Stacking AUC: 0.9953


In [3]:
joblib.dump(stack_model, "/content/drive/MyDrive/SmartRail/stacking_model.pkl")

with open("/content/drive/MyDrive/SmartRail/results.json", "r") as f:
    results = json.load(f)

results["Stacking"] = {"auc": auc_stack}

with open("/content/drive/MyDrive/SmartRail/results.json", "w") as f:
    json.dump(results, f)

print("FINAL RESULTS:")
for k, v in results.items():
    auc_val = v.get("auc") if isinstance(v, dict) else None
    print(f"{k}: AUC={auc_val}")

print("\n✅ ML PORTION OF PROJECT COMPLETE")

FINAL RESULTS:
Logistic Regression: AUC=None
Naive Bayes: AUC=0.9859
KNN: AUC=0.9971
SVM: AUC=0.9993
Decision Tree: AUC=0.9991
Bagging: AUC=0.9999
Random Forest: AUC=1.0
AdaBoost: AUC=0.9999
Gradient Boosting: AUC=1.0
XGBoost: AUC=1.0
XGBoost_final_timesplit: AUC=0.9997
XGBoost_tuned: AUC=0.9999679908500649
Stacking: AUC=0.9952510675813161

✅ ML PORTION OF PROJECT COMPLETE
